# Serve the tweet phrase model from Colab

Starts a small web server with your trained models: a web page where you type a tweet and see the
phrase marked, and an API that other programs can call.

You need the models from `train_on_colab.ipynb` in `tse/work` on Drive (level two is optional).
A GPU runtime answers fastest, but a CPU runtime works too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
import os, glob
PROJECT = '/content/drive/MyDrive/tse'
WORK = f'{PROJECT}/work'
MODELS = f'{WORK}/models'
PRETRAINED = f'{PROJECT}/roberta-base-squad2'
PORT = 8000
API_KEY = ''   # set a password here before sharing a public link
FOLDS = ''     # '' loads every fold; '0' loads one model per seed (starts faster, less memory)

print('Level-one models:', sorted(os.path.basename(f) for f in glob.glob(f'{MODELS}/model_*.bin')))
print('Level-two models:', sorted(os.path.basename(f) for f in glob.glob(f'{WORK}/level2/*.pt')))

In [ ]:
!pip -q install fastapi uvicorn

## Start the server

In [ ]:
import subprocess, sys, time, requests

# Stop a server left over from an earlier start. If it kept running, it would
# answer on the same port with the old code and the new server could not start.
subprocess.run(['pkill', '-f', 'serve.py'])
for _ in range(30):
    try:
        requests.get(f'http://localhost:{PORT}/api/health', timeout=1)
        time.sleep(1)                     # still shutting down
    except requests.exceptions.ConnectionError:
        break

cmd = [sys.executable, 'serve.py', '--pretrained', PRETRAINED, '--work-dir', WORK,
       '--models-dir', MODELS, '--port', str(PORT)]
if FOLDS:
    cmd += ['--folds'] + FOLDS.split()
env = dict(os.environ, TSE_API_KEY=API_KEY)
server = subprocess.Popen(cmd, cwd=PROJECT, env=env,
                          stdout=open('/content/serve.log', 'w'), stderr=subprocess.STDOUT)

# Loading the models takes a minute or two. The first start after an update can
# take a little longer: it builds the score table for the page (once, no training).
ready = False
for _ in range(300):
    if server.poll() is not None:
        break
    try:
        print(requests.get(f'http://localhost:{PORT}/api/health', timeout=2).json())
        ready = True
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
if not ready:
    print(open('/content/serve.log').read()[-3000:])

In [ ]:
# call the API from Python; 'answer' is optional and adds the real Jaccard
headers = {'X-API-Key': API_KEY} if API_KEY else {}
r = requests.post(f'http://localhost:{PORT}/api/extract', headers=headers,
                  json={'text': 'Sooo SAD I will miss you here in San Diego!!!',
                        'sentiment': 'negative', 'answer': 'Sooo SAD'})
r.json()

## Open the web page

This link works only for you, in this browser, while the notebook runs.

In [ ]:
from google.colab.output import eval_js
print(eval_js(f'google.colab.kernel.proxyPort({PORT})'))

## Optional: a public link for colleagues

This uses a free Cloudflare quick tunnel (no account needed). Anyone with the link can use the page and the
API while this notebook runs, so set `API_KEY` above first if you want to limit it (the page then asks for the key).

Colab's rules say it is not meant for running web services, and free sessions can be stopped when they are
used mainly through a web page. Keep this to short demos. For a page that stays online, host the models
elsewhere, for example on a Hugging Face Space.

In [ ]:
import re
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    found = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if found:
        url = found[0]
        break
print('Public link:', url or 'not ready, see /content/tunnel.log')

## Stop

In [ ]:
subprocess.run(['pkill', '-f', 'serve.py'])
subprocess.run(['pkill', '-f', 'cloudflared'])
print('stopped')